### Preprocessing

This is the first step of our BioProv project. We are going to download the necessary data using a **Program**, and once we have the metadata table from such data, we are going to import that as a **Project**.

In [68]:
import bioprov as bp

In [4]:
ncbi_dl = bp.Program("ncbi-genome-download")

params = [
    bp.Parameter("-F", "fasta,assembly-report"),
    bp.Parameter("-s", "genbank"),
    bp.Parameter("-A", "data/genomes.txt"),
    bp.Parameter("-o", "data/"),
    bp.Parameter("-m", "data/metadata.tsv"),
    bp.Parameter("bacteria")
]

for param in params:
    ncbi_dl.add_parameter(param)

In [6]:
ncbi_dl.run()

Running program 'ncbi-genome-download'.
Command is:
/Users/vini/anaconda3/envs/bp-use-case/bin/ncbi-genome-download \ 
	-F fasta,assembly-report \ 
	-s genbank \ 
	-A data/genomes.txt \ 
	-o data/ \ 
	-m data/metadata.tsv \ 
	bacteria


Run of Program 'ncbi-genome-download' with 6 parameter(s).
Started at Mon Jan  4 10:57:40 2021.
Ended at Mon Jan  4 10:59:07 2021.
Status is finished.

The resulting `metadata.tsv` table contains metadata about the files that were just downloaded. We can therefore use it to import our project. However, the table contains duplicated rows (a limitation of the `ncbi-genome-download` package. We are going to run a simple Linux command to get rid of those:

In [70]:
metadata_table = bp.File("data/metadata.tsv", tag="metadata-table")

sort = bp.Program("sort")

params = [
    bp.Parameter("-t", "' '"),
    bp.Parameter("-k", "1,1"),
    bp.Parameter("-u", str(metadata_table)),
    bp.Parameter("-r"),
    bp.Parameter("-o", str(metadata_table))
]

for param in params:
    sort.add_parameter(param)
    
sort.run()

Running program 'sort'.
Command is:
/usr/bin/sort \ 
	-t ' \ 
	' -k \ 
	1,1 -u \ 
	/Users/vini/Bio/BioProv/use-case/data/metadata.tsv -r \ 
	-o /Users/vini/Bio/BioProv/use-case/data/metadata.tsv 


Run of Program 'sort' with 5 parameter(s).
Started at Mon Jan  4 11:23:11 2021.
Ended at Mon Jan  4 11:23:11 2021.
Status is finished.

A second aspect of the `ncbi-genome-download` package is that it downloads the files in a compressed format. We are going to create a program to decompress those files and another one to modify the metadata table accordingly.

In [90]:
gunzip = bp.Program("gunzip")

gunzip.add_parameter(bp.Parameter("data/genbank/bacteria/*/*"))

gunzip.run()

Running program 'gunzip'.
Command is:
/usr/bin/gunzip data/genbank/bacteria/*/*


Run of Program 'gunzip' with 1 parameter(s).
Started at Mon Jan  4 11:29:50 2021.
Ended at Mon Jan  4 11:29:50 2021.
Status is finished.

We can now import the file normally. Since we are creating our project, we can associate the two previous programs with it.

In [79]:
proj = bp.read_csv("data/metadata.tsv",
                   sep="\t",
                   file_cols="local_filename",
                   tag="bp-use-case")

proj.add_programs((ncbi_dl, sort))

In [84]:
proj.samples

{'GCA_003990665.2': Sample 'GCA_003990665.2' with 1 file(s).,
 'GCA_003149585.1': Sample 'GCA_003149585.1' with 1 file(s).,
 'GCA_002252705.1': Sample 'GCA_002252705.1' with 1 file(s).,
 'GCA_001182765.1': Sample 'GCA_001182765.1' with 1 file(s).,
 'GCA_000760375.1': Sample 'GCA_000760375.1' with 1 file(s).,
 'GCA_000316515.1': Sample 'GCA_000316515.1' with 1 file(s).,
 'GCA_000153045.1': Sample 'GCA_000153045.1' with 1 file(s).,
 'GCA_000063525.1': Sample 'GCA_000063525.1' with 1 file(s).,
 'GCA_000063505.1': Sample 'GCA_000063505.1' with 1 file(s).,
 'GCA_000019485.1': Sample 'GCA_000019485.1' with 1 file(s).,
 'GCA_000012625.1': Sample 'GCA_000012625.1' with 1 file(s).,
 'GCA_000011485.1': Sample 'GCA_000011485.1' with 1 file(s).,
 'GCA_000010065.1': Sample 'GCA_000010065.1' with 1 file(s).,
 'GCA_000007925.1': Sample 'GCA_000007925.1' with 1 file(s).}

In [83]:
proj["GCA_003990665.2"].files

{'local_filename': /Users/vini/Bio/BioProv/use-case/data/genbank/bacteria/GCA_003990665.2/GCA_003990665.2_ASM399066v2_genomic.fna.gz}